# Stage 2 · ViT versus U-Net on NYU Depth V2

Select a Colab GPU and run the cells in order. Upload the saved Stage 1 NYU Depth V2 ZIP when prompted. Keep the results ZIP from the final cell.

ViT starts from random weights. Both models use the same paired data, preprocessing, loss, and training budget. The notebook scores both models on the same held-out pixels and times both on this runtime.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import math
import re
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
EPOCHS, PATIENCE, BATCH_SIZE = 30, 6, 4
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
import h5py

HEIGHT, WIDTH = 192, 256
MAX_DEPTH = 10.0
NAME = "nyu"
DATA = Path("/content/stage1/datasets/nyu")
DATA.mkdir(parents=True, exist_ok=True)
source = DATA / "nyu_depth_v2_labeled.mat"
if not source.is_file():
    partial = source.with_suffix(".part")
    subprocess.run(["wget", "-c", "-O", str(partial),
                    "https://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat"],
                   check=True)
    partial.replace(source)

dataset_file = h5py.File(source, "r")
images = dataset_file["images"]
depths = dataset_file["depths"]
assert images.shape == (1449, 3, 640, 480)
assert depths.shape == (1449, 640, 480)
scenes = [
    "".join(chr(int(code)) for code in dataset_file[ref][...].ravel())
    for ref in dataset_file["scenes"][...].ravel()
]
assert len(scenes) == len(images)
records = [(f"nyu_{i:04d}", scene) for i, scene in enumerate(scenes)]

def read_pair(index):
    rgb = Image.fromarray(images[index].transpose(2, 1, 0))
    depth = Image.fromarray(depths[index].T.astype(np.float32))
    return rgb, depth

In [ ]:
from google.colab import files
from io import BytesIO
from zipfile import ZipFile

print("Upload the Stage 1 NYU Depth V2 results ZIP (20261121T211823767832Z.zip)")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one Stage 1 results ZIP.")
with ZipFile(BytesIO(next(iter(uploaded.values())))) as baseline:
    manifest = json.loads(baseline.read("splits.json"))
    original_metrics = json.loads(baseline.read("metrics.json"))
    unet_weights = baseline.read("unet_best.pt")
del uploaded

expected = dict(dataset=NAME, model="U-Net", height=HEIGHT, width=WIDTH,
                max_depth_m=MAX_DEPTH, seed=SEED, epochs=EPOCHS,
                patience=PATIENCE, batch_size=BATCH_SIZE,
                training_loss="masked MAE in meters")
for key, value in expected.items():
    if original_metrics.get(key) != value:
        raise ValueError(f"Stage 1 settings mismatch for {key}. Check the uploaded ZIP.")

split_names = ("train", "validation", "test")
if set(manifest) != set(split_names):
    raise ValueError("Expected train, validation, and test splits.")
indices = {}
seen = set()
seen_groups = set()
for name in split_names:
    rows = manifest[name]
    if not rows:
        raise ValueError(f"Empty {name} split.")
    groups = set()
    indices[name] = []
    for row in rows:
        index = row["index"]
        if not 0 <= index < len(records) or records[index] != (row["sample"], row["group"]):
            raise ValueError(f"Dataset does not match the Stage 1 manifest: {row}")
        if index in seen:
            raise ValueError(f"Duplicate index in Stage 1 manifest: {index}")
        seen.add(index)
        groups.add(row["group"])
        indices[name].append(index)
    if groups & seen_groups:
        raise ValueError(f"Scene or drive overlap in {name} split.")
    seen_groups.update(groups)
if len(seen) != len(records):
    raise ValueError("The current dataset has a different number of pairs than Stage 1.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = Path("/content/stage2/results") / NAME / run_id
output.mkdir(parents=True)
(output / "splits.json").write_text(json.dumps(manifest, indent=2))
print({name: len(rows) for name, rows in indices.items()})

class PairedDepth(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        rgb, depth = read_pair(self.rows[item])
        rgb = rgb.resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR)
        depth = depth.resize((WIDTH, HEIGHT), Image.Resampling.NEAREST)
        x = torch.from_numpy(np.array(rgb, dtype=np.float32)).permute(2, 0, 1) / 255
        y = torch.from_numpy(np.array(depth, dtype=np.float32)).unsqueeze(0)
        return x, y

datasets = {name: PairedDepth(rows) for name, rows in indices.items()}
loaders = {
    name: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(name == "train"),
                     generator=torch.Generator().manual_seed(SEED))
    for name, ds in datasets.items()
}

In [ ]:
class UNetDepth(nn.Module):
    def __init__(self):
        super().__init__()

        def block(inputs, outputs):
            return nn.Sequential(
                nn.Conv2d(inputs, outputs, 3, padding=1), nn.ReLU(),
                nn.Conv2d(outputs, outputs, 3, padding=1), nn.ReLU(),
            )

        self.pool = nn.MaxPool2d(2)
        self.enc1 = block(3, 16)
        self.enc2 = block(16, 32)
        self.enc3 = block(32, 64)
        self.bridge = block(64, 128)
        self.dec3 = block(128 + 64, 64)
        self.dec2 = block(64 + 32, 32)
        self.dec1 = block(32 + 16, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, rgb):
        a = self.enc1(rgb)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        value = self.bridge(self.pool(c))
        for skip, decode in ((c, self.dec3), (b, self.dec2), (a, self.dec1)):
            value = F.interpolate(value, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            value = decode(torch.cat((value, skip), dim=1))
        return F.softplus(self.head(value))

class ViTDepth(nn.Module):
    def __init__(self):
        super().__init__()
        patch_size, dim = 16, 128
        self.grid = (HEIGHT // patch_size, WIDTH // patch_size)
        self.patch_embedding = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.position = nn.Parameter(torch.empty(1, math.prod(self.grid), dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        block = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=4 * dim, dropout=0.1,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            block, num_layers=4, norm=nn.LayerNorm(dim), enable_nested_tensor=False,
        )
        self.depth_head = nn.Sequential(
            nn.Conv2d(dim, 64, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, rgb):
        patches = self.patch_embedding(rgb)
        tokens = patches.flatten(2).transpose(1, 2)
        features = self.encoder(tokens + self.position).transpose(1, 2)
        features = features.reshape(rgb.shape[0], -1, *self.grid)
        depth = self.depth_head(features)
        return F.softplus(F.interpolate(
            depth, size=rgb.shape[-2:], mode="bilinear", align_corners=False,
        ))

unet = UNetDepth().to(device)
unet.load_state_dict(torch.load(BytesIO(unet_weights), map_location=device, weights_only=True))
unet.eval()
torch.manual_seed(SEED)

In [ ]:
def evaluate_epoch(network, batches, optimizer=None):
    network.train(optimizer is not None)
    sums = torch.zeros(4, dtype=torch.float64, device=device)
    count = 0
    with torch.set_grad_enabled(optimizer is not None):
        for rgb, depth in batches:
            rgb, depth = rgb.to(device), depth.to(device)
            valid = torch.isfinite(depth) & (depth > 0) & (depth <= MAX_DEPTH)
            observed = depth[valid]
            if observed.numel() == 0:
                continue
            predicted = network(rgb)[valid]
            error = (predicted - observed).abs()
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                error.mean().backward()
                optimizer.step()
            with torch.no_grad():
                sums += torch.stack((
                    error.sum(dtype=torch.float64),
                    error.square().sum(dtype=torch.float64),
                    (error / observed).sum(dtype=torch.float64),
                    (torch.maximum(predicted / observed, observed / predicted) < 1.25).sum(),
                ))
                count += observed.numel()
    if count == 0:
        raise ValueError("No valid depth pixels.")
    mae, mse, abs_rel, delta1 = (sums / count).cpu().tolist()
    if not all(math.isfinite(value) for value in (mae, mse, abs_rel, delta1)):
        raise FloatingPointError("Non-finite evaluation metrics.")
    return dict(mae_m=mae, rmse_m=math.sqrt(mse), abs_rel=abs_rel, delta1=delta1, pixels=count)

model = ViTDepth().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
best_mae, stale, best_epoch = float("inf"), 0, 0
checkpoint = output / "vit_best.pt"
with (output / "history.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["epoch", "train_mae_m", "val_mae_m", "val_rmse_m"])
    writer.writeheader()
    for epoch in range(1, EPOCHS + 1):
        train = evaluate_epoch(model, loaders["train"], optimizer)
        val = evaluate_epoch(model, loaders["validation"])
        writer.writerow(dict(epoch=epoch, train_mae_m=train["mae_m"],
                             val_mae_m=val["mae_m"], val_rmse_m=val["rmse_m"]))
        stream.flush()
        print(f"Epoch {epoch:02d} | train MAE {train['mae_m']:.3f} m | validation MAE {val['mae_m']:.3f} m")
        if val["mae_m"] < best_mae:
            best_mae, stale, best_epoch = val["mae_m"], 0, epoch
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= PATIENCE:
                break

model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()

In [ ]:
def benchmark(network):
    network.eval()
    sample = datasets["test"][0][0].unsqueeze(0).to(device)
    times = []
    with torch.inference_mode():
        for step in range(30):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            network(sample)
            if device.type == "cuda":
                torch.cuda.synchronize()
            if step >= 10:
                times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))

report = {
    "dataset": NAME,
    "height": HEIGHT, "width": WIDTH, "max_depth_m": MAX_DEPTH,
    "seed": SEED, "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
    "training_loss": "masked MAE in meters",
    "device": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu",
    "models": {},
}
for label, network, epoch in [("U-Net", unet, original_metrics["best_epoch"]),
                              ("ViT", model, best_epoch)]:
    scores = {name: evaluate_epoch(network, loaders[name]) for name in ("validation", "test")}
    for name, metrics in scores.items():
        if metrics["pixels"] != original_metrics[name]["pixels"]:
            raise ValueError(f"{name} pixel count differs from Stage 1.")
    report["models"][label] = dict(best_epoch=epoch, **scores,
        parameters=sum(p.numel() for p in network.parameters()),
        median_forward_ms=benchmark(network))

(output / "comparison.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

In [ ]:
for name, dataset in datasets.items():
    for i in np.linspace(0, len(dataset) - 1, min(3, len(dataset)), dtype=int):
        rgb, depth = dataset[i]
        with torch.inference_mode():
            image = rgb.unsqueeze(0).to(device)
            estimate = model(image)[0, 0].cpu().numpy()
            baseline_depth = unet(image)[0, 0].cpu().numpy()
        measured = depth[0].numpy()
        measured = np.ma.masked_where(
            ~np.isfinite(measured) | (measured <= 0) | (measured > MAX_DEPTH), measured
        )
        fig, axes = plt.subplots(1, 4, figsize=(15, 4), layout="constrained")
        axes[0].imshow(rgb.permute(1, 2, 0))
        axes[1].imshow(measured, cmap="plasma", vmin=0, vmax=MAX_DEPTH)
        axes[2].imshow(baseline_depth, cmap="plasma", vmin=0, vmax=MAX_DEPTH)
        heatmap = axes[3].imshow(estimate, cmap="plasma", vmin=0, vmax=MAX_DEPTH)
        for axis, title in zip(axes, ["RGB", "Reference depth", "U-Net depth", "ViT depth"]):
            axis.set_title(title)
            axis.axis("off")
        fig.colorbar(heatmap, ax=axes[1:], label="Depth (m)", extend="max")
        fig.savefig(output / f"{name}_{i:04d}.png", dpi=140)
        plt.show()
        plt.close(fig)

from google.colab import files
archive = shutil.make_archive(str(output), "zip", root_dir=output)
files.download(archive)